# Dataset Description
## Department with the highest and lowest Church rate per 100k inhabitants

**Source**: [El Mapa de Fe](https://encuentraiglesias.com/mapa-de-la-fe-colombia)

# Code
## Imports

In [43]:
import pandas as pd
import matplotlib.pyplot as plt
import re

pd.set_option("display.float_format", lambda x: "%.2f" % x)

## Load data

In [44]:
path = "./data/mapa-fe-colombia.csv"
pop_path = "./data/departamental.csv"

def clean(value):
    buffer = value.lower()
    return re.sub(r"[.,]", "", buffer)

df = pd.read_csv(
    path, encoding="utf-8",
    converters={"estado": clean}
)
pop = pd.read_csv(
    pop_path, encoding="utf-8",
    usecols=[1,2, 308],
    converters={"DPNOM": clean}
)

## Clean data

In [45]:
df["count"] = 1
df_f = (
    df[
        df["denominacion"].isin(["Católica", "Cristiana"])
    ]
    .drop(columns=["municipio", "denominacion", "templo", "lat", "lon"])
    .rename(columns={"estado": "DPNOM"})
)

In [49]:
df_ff = df_f.groupby("DPNOM").sum().reset_index()

,DPNOM,count
0,amazonas,6
1,antioquia,680
2,arauca,9
3,atlántico,165
4,bogotá dc,472
5,bolívar,168
6,boyacá,278
7,caldas,100
8,caquetá,37
9,casanare,56


In [46]:
pop_f = pop[pop["AÑO"].eq(2020)].drop(columns=["AÑO"])

In [58]:
pop_f["DPNOM"] = (
    pop_f["DPNOM"]
    .str.replace(
        "archipiélago de san andrés providencia y santa catalina",
        "san andrés y providencia"
    )
)

In [79]:
final = pd.merge(df_ff, pop_f, on="DPNOM", how="left")
final["rate"] = (final["count"] / final["Total"]) * 100000

print(
    "Top 5:\n",
    final.nlargest(5, columns=["rate"])[["DPNOM", "rate"]],
    "\n\nBottom 5:\n",
    final.nsmallest(5, columns=["rate"])[["DPNOM", "rate"]],
    "\n\nMedian:\n",
    final["rate"].median()
)

Top 5:
                        DPNOM  rate
26  san andrés y providencia 35.41
6                     boyacá 22.26
24                   quindío 15.31
27                 santander 14.82
9                   casanare 12.68 

Bottom 5:
         DPNOM  rate
14    córdoba  2.26
11      cesar  2.75
2      arauca  3.33
19  magdalena  4.26
12      chocó  4.32 

Median:
 7.808647240177721
